# 01 - Simulated systems and extremes

Research question: how do extreme events and clustering differ across stochastic baselines and nonlinear dynamical systems?

Data dependencies: none. This notebook uses only synthetic simulations.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from dynsys_econometrics.extremes import estimate_runs_extremal_index
from dynsys_econometrics.simulation import (
    logistic_map,
    simulate_ar1,
    simulate_garch11,
    simulate_iid_gaussian,
    simulate_pomeau_manneville,
    simulate_student_t,
)


In [ ]:
n_steps = 4000
threshold_quantile = 0.97

series = {
    "iid_gaussian": simulate_iid_gaussian(n_steps, seed=7),
    "student_t": simulate_student_t(n_steps, df=6, seed=7),
    "ar1": simulate_ar1(n_steps, phi=0.7, sigma=1.0, seed=7),
    "garch11": simulate_garch11(n_steps, seed=7),
    "logistic": logistic_map(n_steps, x0=0.23, r=4.0),
    "pomeau_manneville": simulate_pomeau_manneville(n_steps, x0=1e-3, a=1.0, z=1.5),
}

summary_rows = []
for name, values in series.items():
    result = estimate_runs_extremal_index(values, threshold_quantile=threshold_quantile, run_length=3)
    summary_rows.append(
        {
            "series": name,
            "theta_runs": result.theta_hat,
            "n_exceedances": result.n_exceedances,
            "n_clusters": result.n_clusters,
        }
    )

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
output_dir = Path("article/figures")
output_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(2, 1, figsize=(11, 8))
pd.DataFrame(series)[["iid_gaussian", "ar1", "garch11"]].iloc[:500].plot(ax=axes[0], alpha=0.8)
axes[0].set_title("Selected simulated observables")
axes[0].set_xlabel("time")
axes[0].set_ylabel("value")

summary.plot(x="series", y="theta_runs", kind="bar", legend=False, ax=axes[1], color="#4c78a8")
axes[1].set_title("Runs extremal index by generator")
axes[1].set_xlabel("series")
axes[1].set_ylabel("theta")

fig.tight_layout()
fig.savefig(output_dir / "01_simulated_systems_extremes.png", dpi=160)
plt.close(fig)
str(output_dir / "01_simulated_systems_extremes.png")
